In [2]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
!pip install dotenv

  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
Using cached python_dotenv-1.1.1-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dotenv]


### To run the Agent workflow

In [4]:
from src.hello.ml.exception.custom_exception import MultiAgentWorkflowException
from src.hello.ml.logger import GLOBAL_LOGGER as logger
from src.hello.ml.multi_agent_workflow.agents_workflow import AgentsWorkflow

workflow = AgentsWorkflow().build_workflow()

input_data = """
[{'Quarter': 'Q2 2022', 'Overall': 18.01, 'Class A': 21.1, 'Class B': 17.0},
{'Quarter': 'Q3 2022', 'Overall': 17.7, 'Class A': 20.8, 'Class B': 16.6},
{'Quarter': 'Q4 2022', 'Overall': 17.49, 'Class A': 19.7, 'Class B': 17.2},
{'Quarter': 'Q1 2023', 'Overall': 17.6, 'Class A': 20.4, 'Class B': 16.8},
{'Quarter': 'Q2 2023', 'Overall': 17.72, 'Class A': 20.4, 'Class B': 17.1},
{'Quarter': 'Q3 2023', 'Overall': 17.99, 'Class A': 21.1, 'Class B': 17.1},
{'Quarter': 'Q4 2023', 'Overall': 17.95, 'Class A': 21.0, 'Class B': 17.2},
{'Quarter': 'Q1 2024', 'Overall': 18.36, 'Class A': 20.9, 'Class B': 18.1},
{'Quarter': 'Q2 2024', 'Overall': 18.96, 'Class A': 22.3, 'Class B': 17.9},
{'Quarter': 'Q3 2024', 'Overall': 19.19, 'Class A': 22.0, 'Class B': 18.7},
{'Quarter': 'Q4 2024', 'Overall': 19.13, 'Class A': 21.8, 'Class B': 18.8},
{'Quarter': 'Q1 2025', 'Overall': 18.65, 'Class A': 21.0, 'Class B': 18.5},
{'Quarter': 'Q2 2025', 'Overall': 18.29, 'Class A': 20.4, 'Class B': 18.4}]
"""

app = workflow.compile()

ModuleNotFoundError: No module named 'src'

In [ ]:
app

In [ ]:

result = app.invoke({"session_type": "vacancy", "input_data": input_data})  # type: ignore
print(result["summary_result"])

## For running the Summary Agent alone

In [5]:
from src.hello.ml.agents.session_summary_agent import SessionSummaryAgent

state = {'messages': [], 'session_type': 'vacancy', 'input_data': "\n    [{'Quarter': 'Q2 2022', 'Overall': 18.01, 'Class A': 21.1, 'Class B': 17.0},\n    {'Quarter': 'Q3 2022', 'Overall': 17.7, 'Class A': 20.8, 'Class B': 16.6},\n    {'Quarter': 'Q4 2022', 'Overall': 17.49, 'Class A': 19.7, 'Class B': 17.2},\n    {'Quarter': 'Q1 2023', 'Overall': 17.6, 'Class A': 20.4, 'Class B': 16.8},\n    {'Quarter': 'Q2 2023', 'Overall': 17.72, 'Class A': 20.4, 'Class B': 17.1},\n    {'Quarter': 'Q3 2023', 'Overall': 17.99, 'Class A': 21.1, 'Class B': 17.1},\n    {'Quarter': 'Q4 2023', 'Overall': 17.95, 'Class A': 21.0, 'Class B': 17.2},\n    {'Quarter': 'Q1 2024', 'Overall': 18.36, 'Class A': 20.9, 'Class B': 18.1},\n    {'Quarter': 'Q2 2024', 'Overall': 18.96, 'Class A': 22.3, 'Class B': 17.9},\n    {'Quarter': 'Q3 2024', 'Overall': 19.19, 'Class A': 22.0, 'Class B': 18.7},\n    {'Quarter': 'Q4 2024', 'Overall': 19.13, 'Class A': 21.8, 'Class B': 18.8},\n    {'Quarter': 'Q1 2025', 'Overall': 18.65, 'Class A': 21.0, 'Class B': 18.5},\n    {'Quarter': 'Q2 2025', 'Overall': 18.29, 'Class A': 20.4, 'Class B': 18.4}]\n    ", 'summary_agent_retries': 1, 'unit_check_agent_retries': 0, 'data_check_agent_retries': 0, 'validation_agent_retries': 0, 'improvement_feedback': ''}

summary = SessionSummaryAgent(dict(state)).generate_summary()

ModuleNotFoundError: No module named 'src'

In [ ]:
print(summary.content)

### For experimenting the module with individual components

In [17]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate

# llm = ChatOpenAI(model="gpt-5", temperature=0, reasoning_effort="high")

llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [18]:
from langchain.agents import AgentExecutor, create_react_agent

from hello.ml.utils.config_loader import load_prompt_from_config


In [19]:
# Create ReAct prompt template that incorporates the loaded prompt and feedback
react_prompt = PromptTemplate.from_template("""
You are a ReAct agent specialized in creating comprehensive summaries of session data.

Your task is to process the input data according to the specific prompt template provided and generate a summary.

Original Prompt Template: {original_prompt}

{feedback_instructions}

You have no external tools available. You must use your reasoning capabilities to analyze the data step by step and create a summary that follows the format and requirements specified in the original prompt template.

Use the following format:

Question: the input data you must summarize according to the prompt template
Thought: you should always think about what analysis step to take next
Action: Since no tools are available, describe what analysis you would perform
Action Input: The data you're analyzing
Observation: Your analysis and findings from examining the data
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now have enough information to create a comprehensive summary
Final Answer: A well-structured summary that follows the original prompt template format

Begin!

Question: Create a summary of this session data using the provided template:

Input Data: {input_json}
Few Shot Examples: {few_shot_examples}

Ignore these placeholders: 
{agent_scratchpad}
{tools}
{tool_names}

Use the original prompt template to guide your summary format and content requirements.

Thought:""")

In [20]:
print(react_prompt)

input_variables=['agent_scratchpad', 'feedback_instructions', 'few_shot_examples', 'input_json', 'original_prompt', 'tool_names', 'tools'] input_types={} partial_variables={} template="\nYou are a ReAct agent specialized in creating comprehensive summaries of session data.\n\nYour task is to process the input data according to the specific prompt template provided and generate a summary.\n\nOriginal Prompt Template: {original_prompt}\n\n{feedback_instructions}\n\nYou have no external tools available. You must use your reasoning capabilities to analyze the data step by step and create a summary that follows the format and requirements specified in the original prompt template.\n\nUse the following format:\n\nQuestion: the input data you must summarize according to the prompt template\nThought: you should always think about what analysis step to take next\nAction: Since no tools are available, describe what analysis you would perform\nAction Input: The data you're analyzing\nObservation:

In [21]:
input_data = """
[{'Quarter': 'Q2 2022', 'Overall': 18.01, 'Class A': 21.1, 'Class B': 17.0},
{'Quarter': 'Q3 2022', 'Overall': 17.7, 'Class A': 20.8, 'Class B': 16.6},
{'Quarter': 'Q4 2022', 'Overall': 17.49, 'Class A': 19.7, 'Class B': 17.2},
{'Quarter': 'Q1 2023', 'Overall': 17.6, 'Class A': 20.4, 'Class B': 16.8},
{'Quarter': 'Q2 2023', 'Overall': 17.72, 'Class A': 20.4, 'Class B': 17.1},
{'Quarter': 'Q3 2023', 'Overall': 17.99, 'Class A': 21.1, 'Class B': 17.1},
{'Quarter': 'Q4 2023', 'Overall': 17.95, 'Class A': 21.0, 'Class B': 17.2},
{'Quarter': 'Q1 2024', 'Overall': 18.36, 'Class A': 20.9, 'Class B': 18.1},
{'Quarter': 'Q2 2024', 'Overall': 18.96, 'Class A': 22.3, 'Class B': 17.9},
{'Quarter': 'Q3 2024', 'Overall': 19.19, 'Class A': 22.0, 'Class B': 18.7},
{'Quarter': 'Q4 2024', 'Overall': 19.13, 'Class A': 21.8, 'Class B': 18.8},
{'Quarter': 'Q1 2025', 'Overall': 18.65, 'Class A': 21.0, 'Class B': 18.5},
{'Quarter': 'Q2 2025', 'Overall': 18.29, 'Class A': 20.4, 'Class B': 18.4}]
"""


In [22]:
original_prompt, few_shot_examples = load_prompt_from_config("vacancy")

{"timestamp": "2025-09-23T16:06:34.459200Z", "level": "info", "logger": "multi_agent_workflow", "pathname": "/Users/saikumar.talla/Workspace/CBRE/CBRE_repo/cbre-app/backend/src/hello/ml/utils/config_loader.py", "lineno": 47, "func_name": "load_prompt_from_config", "event": "Prompt version: v2"}
{"timestamp": "2025-09-23T16:06:34.460425Z", "level": "info", "logger": "multi_agent_workflow", "pathname": "/Users/saikumar.talla/Workspace/CBRE/CBRE_repo/cbre-app/backend/src/hello/ml/utils/config_loader.py", "lineno": 55, "func_name": "load_prompt_from_config", "event": "Loading prompt from: /Users/saikumar.talla/Workspace/CBRE/CBRE_repo/cbre-app/backend/src/hello/ml/config/prompts_v2.yaml"}
{"timestamp": "2025-09-23T16:06:34.474872Z", "level": "info", "logger": "multi_agent_workflow", "pathname": "/Users/saikumar.talla/Workspace/CBRE/CBRE_repo/cbre-app/backend/src/hello/ml/utils/config_loader.py", "lineno": 65, "func_name": "load_prompt_from_config", "event": "Loaded prompt for session type:

In [24]:
agent = create_react_agent(llm = llm, tools=[], prompt=react_prompt)

In [27]:
agent_executor = AgentExecutor(
            agent=agent,
            tools=[],  # No tools
            verbose= True, 
            handle_parsing_errors=True,
            max_iterations=8,
            # early_stopping_method="generate",
        )

In [28]:
agent_executor.invoke(
                {
                    "original_prompt": original_prompt,
                    "input_json": input_data,
                    "few_shot_examples": few_shot_examples,
                    "feedback_instructions": {},
                }
)



> Entering new AgentExecutor chain...


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


To create a summary, I need to analyze the provided data and align it with the style and structure of the few-shot examples. The data includes quarterly vacancy rates for overall, Class A, and Class B spaces from Q2 2022 to Q2 2025. I will calculate the changes in vacancy rates over the last quarter, year, and three years, as demonstrated in the examples.

Action: Analyze the data to calculate the changes in vacancy rates over the last quarter, year, and three years.

Action Input: 
- Latest quarter: Q2 2025
- Previous quarter: Q1 2025
- Year-over-year comparison: Q2 2024
- Three-year comparison: Q2 2022
Analyze the data to calculate the changes in vacancy rates over the last quarter, year, and three years. is not a valid tool, try one of [].

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


To create a summary, I need to analyze the provided data and align it with the style and structure of the few-shot examples. The data includes quarterly vacancy rates for overall, Class A, and Class B spaces from Q2 2022 to Q2 2025. I will calculate the changes in vacancy rates over the last quarter, year, and three years, as demonstrated in the examples.

Action: Calculate the changes in vacancy rates over the last quarter, year, and three years.

Action Input: 
- Latest quarter: Q2 2025
- Previous quarter: Q1 2025
- Year-over-year comparison: Q2 2024
- Three-year comparison: Q2 2022
Calculate the changes in vacancy rates over the last quarter, year, and three years. is not a valid tool, try one of [].

...

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


To create a summary, I need to analyze the provided data and align it with the style and structure of the few-shot examples. The data includes quarterly vacancy rates for overall, Class A, and Class B spaces from Q2 2022 to Q2 2025. I will calculate the changes in vacancy rates over the last quarter, year, and three years, as demonstrated in the examples.

Action: Calculate the changes in vacancy rates over the last quarter, year, and three years.

Action Input: 
- Latest quarter: Q2 2025
- Previous quarter: Q1 2025
- Year-over-year comparison: Q2 2024
- Three-year comparison: Q2 2022
Calculate the changes in vacancy rates over the last quarter, year, and three years. is not a valid tool, try one of [].

HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


To create a summary, I need to analyze the provided data and align it with the style and structure of the few-shot examples. The data includes quarterly vacancy rates for overall, Class A, and Class B spaces from Q2 2022 to Q2 2025. I will calculate the changes in vacancy rates over the last quarter, year, and three years, as demonstrated in the examples.

Action: Calculate the changes in vacancy rates over the last quarter, year, and three years.

Action Input: 
- Latest quarter: Q2 2025
- Previous quarter: Q1 2025
- Year-over-year comparison: Q2 2024
- Three-year comparison: Q2 2022
Calculate the changes in vacancy rates over the last quarter, year, and three years. is not a valid tool, try one of [].

> Finished chain.


{'original_prompt': 'You are an expert commercial real estate analyst tasked with generating professional "VACANCY ANALYSIS" reports. Your writing must strictly follow the style, tone, and structure demonstrated in the provided few-shot examples, ensuring outputs resemble brokerage research publications.\n\n## TASK OVERVIEW\nYou will be provided with a vacancy data in structured JSON format. Your job is to write a concise, professional overview that mirrors the examples provided.\n\nSOURCE OF TRUTH\n- Use only the provided few shot examples (if provided) as your source.\n- Do not invent facts or add external commentary.\n- If a metric cannot be computed from the inputs, omit it silently (do not mention missing data).  \nRUNTIME Inputs:\nYou will be provided with the following inputs:\n- Title: The title of the overview.\n- Units: The unit of measurement (e.g., "Sq. Ft. millions", "Sq. Ft.") or blank if not provided.\n- one or more named tables (any count) data related to vacancy.\n- fe